In [ ]:
import pandas as pd
import numpy as np

# ------------ Load the data from raw imports ------------ 
crsp = pd.read_csv('data/wrds_data.csv') 
hmi = pd.read_excel('data/hmi.xls', header=None) 
treasury = pd.read_csv('data/10_year_treasury.csv') 
# ------------  Data Cleaning and Aggregation ------------ 
data_start_idx = hmi[hmi[0] == 1985].index[0]
hmi = hmi.iloc[data_start_idx:].reset_index(drop=True)
hmi.columns = ['Year', 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
hmi['Year'] = hmi['Year'].astype(int)
hmi = hmi.melt(id_vars=['Year'], var_name='Month', value_name='HMI') # row per year -> row per month per year
hmi = hmi.sort_values(['Year', 'Month']).reset_index(drop=True)
hmi['HMI_lag1'] = hmi['HMI'].shift(1)
hmi = hmi.dropna(subset=['HMI_lag1']).reset_index(drop=True)

treasury['date'] = pd.to_datetime(treasury['observation_date'], format='%Y-%m-%d')
treasury['Month'] = treasury['date'].dt.month
treasury['Year'] = treasury['date'].dt.year
treasury = treasury.groupby(['Year', 'Month'])['DGS10'].mean().reset_index() # from daily to monthly

crsp['date'] = pd.to_datetime(crsp['date'])
crsp['Month'] = crsp['date'].dt.month
crsp['Year'] = crsp['date'].dt.year
crsp['RET'] = pd.to_numeric(crsp['RET'], errors='coerce')
crsp['PRC'] = pd.to_numeric(crsp['PRC'], errors='coerce')
crsp = crsp.dropna(subset=['RET', 'PRC']).copy()
# Amihud (2002): ILLIQ = |R| / (|PRC| * VOL); NaN on no-trade days (VOL=0 or approx 0)
# so they are excluded from the monthly mean but still contribute to the return compounding
crsp['illiq_daily'] = np.where(
    crsp['VOL'] > 0,
    crsp['RET'].abs() / (crsp['PRC'].abs() * crsp['VOL']),
    np.nan
)
def calculate_monthly_ret(series):
    return (series + 1).prod() - 1
df_monthly = crsp.groupby(['TICKER', 'Year', 'Month']).agg({
    'illiq_daily': 'mean',
    'RET': calculate_monthly_ret,    
    'VOL': 'sum'                     
}).reset_index()
df_monthly.rename(columns={'RET': 'monthly_ret', 'illiq_daily': 'illiq_avg'}, inplace=True)

# ------------ Merging and Final Clean up------------ 
df = df_monthly.merge(hmi[['Year', 'Month', 'HMI_lag1']], on=['Year', 'Month'], how='inner')
df = df.merge(treasury, on=['Year', 'Month'], how='inner')
df['log_illiq'] = np.log(df['illiq_avg'])
df = df.sort_values(['TICKER', 'Year', 'Month']).reset_index(drop=True)
treatment_tickers = ['ESS', 'AVB', 'INVH', 'UDR', 'MAA', 'CPT', 'AMH']
control_tickers = ['WELL', 'AMT', 'EQIX', 'PLD']
df['treatment'] = df['TICKER'].apply(lambda x: 1 if x in treatment_tickers else (0 if x in control_tickers else np.nan))
df.columns = df.columns.str.lower()
df.to_csv('processed_data.csv', index=False)
# mb some of the stocks had a ticker change/ m&a? AMB, HCN etc. weren't part of the query // df[df['treatment'].isna()]['ticker'].unique()

c:\Users\dvolosh\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [10]:
df['illiq_avg'].describe()

count    3.700000e+03
mean     5.346027e-08
std      6.143113e-07
min      0.000000e+00
25%      1.128337e-10
50%      4.055601e-10
75%      4.036910e-09
max      1.556570e-05
Name: illiq_avg, dtype: float64

In [11]:
# The earliest date for all tickers with non-na treatment status
df[df['treatment'].notna()].groupby('ticker')['year'].min()
# Seems to make sense to drop INVNH since data available only from 2017; mb EQIX too

ticker
AMH     1990
AMT     1990
CPT     1990
EQIX    2000
ESS     1994
INVH    2017
MAA     1994
PLD     1998
UDR     1990
WELL    1993
Name: year, dtype: int32